In [3]:

import ast
import io
import os
import zipfile
import pandas as pd
from collections import Counter
from google.colab import files

# =====================================================
# 1. UPLOAD
# =====================================================

print("Upload CycPeptMPDB_Peptide_All.csv")
uploaded = files.upload()
filename = next(iter(uploaded))

df = pd.read_csv(io.BytesIO(uploaded[filename]),
                 low_memory=False, encoding="utf-8-sig")
del uploaded

print("Total records:", len(df))
print("Columns:", list(df.columns))

# =====================================================
# 2. FIND COLUMNS
# =====================================================

def find_col(keyword):
    matches = [c for c in df.columns
               if keyword.lower() in c.lower()]
    return matches[0] if matches else None

SEQ_COL = next(
    (c for c in df.columns if c.strip().lower() == "sequence"),
    None
)
if SEQ_COL is None:
    raise ValueError("Sequence column not found.")

SHAPE_COL = find_col("Molecule_Shape")
SMILES_COL = find_col("SMILES")
HELM_COL = find_col("HELM")
ID_COL = find_col("CycPeptMPDB_ID")

# =====================================================
# 3. RESIDUE-IDENTITY MAPPING
# =====================================================

AA = set("ACDEFGHIKLMNPQRSTVWY")

def map_residue(token):
    if not isinstance(token, str):
        return None
    token = token.strip()

    if token in AA:
        return token

    if len(token) == 2 and token.startswith("d"):
        if token[1] in AA:
            return token[1]

    for prefix in ("Me_d", "me_d", "me", "Me"):
        if token.startswith(prefix):
            rest = token[len(prefix):]
            if rest in AA:
                return rest

    return None

def parse_residues(value):
    try:
        x = ast.literal_eval(str(value))
        if isinstance(x, (list, tuple)) and x:
            if all(isinstance(a, str) for a in x):
                return x
    except (SyntaxError, ValueError):
        pass
    return None

def convert(value):
    residues = parse_residues(value)
    if residues is None:
        return pd.Series([None, "parse_error", "", None])

    mapped = [map_residue(x) for x in residues]
    unknown = sorted(set(
        x for x, y in zip(residues, mapped) if y is None
    ))

    if unknown:
        return pd.Series([
            None, "unmapped_monomer",
            "; ".join(unknown), len(residues)
        ])

    return pd.Series([
        "".join(mapped), "mapped", "", len(residues)
    ])

df[[
    "Mapped_Sequence", "Mapping_Status",
    "Unmapped_Monomers", "Original_Monomer_Count"
]] = df[SEQ_COL].apply(convert)

valid = df[df["Mapping_Status"] == "mapped"].copy()

# =====================================================
# 4. ROTATIONAL CANONICALIZATION
# =====================================================

def canonical_rotation(seq):
    return min(
        seq[i:] + seq[:i]
        for i in range(len(seq))
    )

# IMPORTANT:
# 'Circle' alone does not prove head-to-tail closure.
# Therefore automatic rotation is only used for a
# sequence-level sensitivity analysis, NOT as a
# confirmed chemical deduplication.

valid["Rotation_Canonical_Sequence"] = (
    valid["Mapped_Sequence"].apply(canonical_rotation)
)

# Exact string-level uniqueness
exact_unique = valid.drop_duplicates(
    subset=["Mapped_Sequence"]
).copy()

# Rotation-equivalence candidates, regardless of topology.
# This is DIAGNOSTIC ONLY until linkage is verified.
rotation_candidates = valid.drop_duplicates(
    subset=["Rotation_Canonical_Sequence"]
).copy()

# =====================================================
# 5. COUNT ROTATIONAL EQUIVALENCE
# =====================================================

rot_groups = (
    valid.groupby("Rotation_Canonical_Sequence")
    .agg(
        n_rows=("Mapped_Sequence", "size"),
        n_exact_strings=("Mapped_Sequence", "nunique")
    )
    .reset_index()
)

rot_groups = rot_groups[
    rot_groups["n_exact_strings"] > 1
].copy()

# =====================================================
# 6. OUTPUT CSV
# =====================================================

valid.to_csv(
    "CycPeptMPDB_all_mapped_records.csv",
    index=False, encoding="utf-8-sig"
)

exact_unique.to_csv(
    "CycPeptMPDB_exact_unique.csv",
    index=False, encoding="utf-8-sig"
)

rotation_candidates.to_csv(
    "CycPeptMPDB_rotation_candidates.csv",
    index=False, encoding="utf-8-sig"
)

rot_groups.to_csv(
    "CycPeptMPDB_rotation_groups_audit.csv",
    index=False, encoding="utf-8-sig"
)

df[df["Mapping_Status"] != "mapped"].to_csv(
    "CycPeptMPDB_excluded_records.csv",
    index=False, encoding="utf-8-sig"
)

# =====================================================
# 7. FASTA
# =====================================================

with open("CycPeptMPDB_exact_unique.fasta", "w") as f:
    for i, seq in enumerate(
        exact_unique["Mapped_Sequence"], start=1
    ):
        f.write(f">CycPeptMPDB_{i:06d}\n{seq}\n")

# Diagnostic FASTA only; rotational equivalence must
# be verified against molecular topology before use.
with open("CycPeptMPDB_rotation_candidates.fasta", "w") as f:
    for i, seq in enumerate(
        rotation_candidates["Rotation_Canonical_Sequence"],
        start=1
    ):
        f.write(f">CycPeptMPDB_ROT_{i:06d}\n{seq}\n")

# =====================================================
# 8. REPORT
# =====================================================

report = pd.DataFrame([
    ["Total records", len(df)],
    ["Fully mappable records", len(valid)],
    ["Excluded records", len(df) - len(valid)],
    ["Unique exact mapped sequences", len(exact_unique)],
    ["Rotation-equivalent groups (diagnostic)", len(rot_groups)],
    ["Unique after rotation (diagnostic)",
     len(rotation_candidates)]
], columns=["Step", "Count"])

report.to_csv(
    "CycPeptMPDB_filter_report.csv",
    index=False
)

print("\n========== REPORT ==========")
print(report.to_string(index=False))

print("\nStructural columns:")
print("Shape:", SHAPE_COL)
print("SMILES:", SMILES_COL)
print("HELM:", HELM_COL)
print("ID:", ID_COL)

print("\nIMPORTANT:")
print("Rotation-based counts are diagnostic, not yet")
print("chemically confirmed deduplication results.")

# =====================================================
# 9. PACKAGE RESULTS
# =====================================================

outputs = [
    "CycPeptMPDB_all_mapped_records.csv",
    "CycPeptMPDB_exact_unique.csv",
    "CycPeptMPDB_rotation_candidates.csv",
    "CycPeptMPDB_rotation_groups_audit.csv",
    "CycPeptMPDB_excluded_records.csv",
    "CycPeptMPDB_exact_unique.fasta",
    "CycPeptMPDB_rotation_candidates.fasta",
    "CycPeptMPDB_filter_report.csv"
]

zip_path = "CycPeptMPDB_revised_results.zip"

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as zf:
    for filename in outputs:
        zf.write(filename)

files.download(zip_path)


Upload CycPeptMPDB_Peptide_All.csv


Saving CycPeptMPDB_Peptide_All.csv to CycPeptMPDB_Peptide_All (1).csv
Total records: 7451
Columns: ['CycPeptMPDB_ID', 'Source', 'Year', 'Original_Name_in_Source_Literature', 'Structurally_Unique_ID', 'Same_Peptides_ID', 'Same_Peptides_Source', 'Same_Peptides_Permeability', 'Same_Peptides_Assay', 'SMILES', 'HELM', 'HELM_URL', 'Sequence', 'Sequence_LogP', 'Sequence_TPSA', 'Monomer_Length', 'Monomer_Length_in_Main_Chain', 'Molecule_Shape', 'Permeability', 'PAMPA', 'Caco2', 'MDCK', 'RRCK', 'Detection_Limit_1', 'Detection_Limit_2', 'R_PAMAP', 'R_Caco2', 'R_MDCK', 'R_RRCK', 'T_PAMPA', 'EPSA', 'PSA', '_3DPSA', 'NULL', 'MaxEStateIndex', 'MinEStateIndex', 'MaxAbsEStateIndex', 'MinAbsEStateIndex', 'qed', 'MolWt', 'HeavyAtomMolWt', 'ExactMolWt', 'NumValenceElectrons', 'NumRadicalElectrons', 'MaxPartialCharge', 'MinPartialCharge', 'MaxAbsPartialCharge', 'MinAbsPartialCharge', 'FpDensityMorgan1', 'FpDensityMorgan2', 'FpDensityMorgan3', 'BCUT2D_MWHI', 'BCUT2D_MWLOW', 'BCUT2D_CHGHI', 'BCUT2D_CHGLO', 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>